[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JanNikolaidovic/python-text-analytics-lab/blob/main/labs/lab14_pandas_grouping_dv/lab14_student.ipynb)

# Lab 14 — pandas III: Grouping and the Dependent Variable

### Digital Skills and Business Decisioning Using Python

You can now load, explore, and clean a dataset. This lab is about **summarising** it: turning 22,000 rows into a handful of numbers that answer questions. The main tool is **`groupby()`**, which computes a statistic separately for each group (each department, each star rating, each age group) in one line.

It's also where the course reaches its central idea. Every analysis is built around a **dependent variable** (DV): the outcome you want to understand. Choosing it is a decision, and a good choice makes everything that follows (the questions, the charts, the conclusions) clearer. By the end of this lab you'll have chosen a DV for the reviews dataset, checked its properties, and found your first variables that relate to it.

In this lab you'll learn to:

* describe numeric columns with **descriptive statistics**, and recognise a **skewed** distribution,
* **group and aggregate** with `groupby()` and `.agg()`,
* build **crosstabs** of two categorical columns with `pd.crosstab()`,
* turn a number into categories with **`pd.cut()`**,
* explain what a **dependent variable** is, compare candidate DVs, and check **class balance**, and
* compare other variables **across the levels of the DV**.

### Setup

The function below repeats the cleaning steps from Lab 13 (Exercise 8), plus a few of the text features you built there. Starting every notebook from the original file with one function call makes the analysis **reproducible**: anyone can rerun it from the top and get the same results.

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/JanNikolaidovic/python-text-analytics-lab/main/data/womens_clothing_ecommerce_reviews.csv"

def load_reviews(url):
    """Load the clothing reviews and apply the cleaning steps from Lab 13."""
    reviews = pd.read_csv(url, index_col=0)
    reviews.columns = [name.lower().replace(" ", "_") for name in reviews.columns]
    reviews = reviews.rename(columns={"recommended_ind": "recommended",
                                      "division_name": "division",
                                      "department_name": "department"})
    reviews = reviews.dropna(subset=["review_text", "division", "department", "class_name"])
    reviews["title"] = reviews["title"].fillna("")
    reviews = reviews.drop_duplicates(subset=["review_text"])
    reviews["division"] = reviews["division"].replace({"Initmates": "Intimates"})
    reviews["recommended"] = reviews["recommended"].astype(bool)
    reviews["review_text"] = reviews["review_text"].str.replace(r"\s+", " ", regex=True).str.strip()

    # features from the text (Labs 09 and 13)
    reviews["n_words"] = reviews["review_text"].str.split().str.len()
    reviews["n_exclaims"] = reviews["review_text"].str.count("!")
    reviews["mentions_return"] = reviews["review_text"].str.contains(r"\breturn\w*", case=False)
    height = reviews["review_text"].str.extract(r"(\d)'\s?(\d{1,2})").astype(float)
    reviews["height_cm"] = ((height[0] * 12 + height[1]) * 2.54).round()
    reviews["sentiment"] = reviews["rating"].map({1: "negative", 2: "negative", 3: "neutral",
                                                  4: "positive", 5: "positive"})
    return reviews


df = load_reviews(DATA_URL)
print(df.shape)
df.head()

### Part 1: Descriptive Statistics

You've met most of these in NumPy and in `describe()`. Here they are one at a time, on the number of words per review:

In [ ]:
words = df["n_words"]

print(f"mean   {words.mean():.1f}")
print(f"median {words.median():.1f}")
print(f"std    {words.std():.1f}")
print(f"min    {words.min()}   max {words.max()}")
print(words.quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

**`.quantile(q)`** gives the value below which a share `q` of the data falls. The 0.1 quantile is about 22 words: 10% of reviews are shorter than that. The 0.25, 0.5, and 0.75 quantiles are the quartiles from `describe()`, and the 0.5 quantile is the median.

For review length, the mean and median are almost equal: a balanced (**symmetric**) distribution. Now compare the helpful-vote count:

In [ ]:
votes = df["positive_feedback_count"]

print(f"mean   {votes.mean():.2f}")
print(f"median {votes.median():.1f}")
print(f"share with 0 votes: {(votes == 0).mean():.1%}")
print(votes.quantile([0.5, 0.9, 0.99]))

Here the **mean (2.5) is far above the median (1)**. Nearly half the reviews got no votes at all, while 1% got more than about 27, and a few got over 100. Those few large values pull the mean up. A distribution with a long tail of large values like this is called **right-skewed**, and it's extremely common: incomes, house prices, social media likes, and helpful votes all look like this.

> **For skewed data, report the median** (and perhaps the share of zeros). The mean of 2.5 votes describes almost nobody: most reviews got fewer, and a few got many more.

### Practice: Descriptive Statistics

**Exercise 1: The Age Distribution (easy)**

Print the mean, median, and the 0.1 and 0.9 quantiles of `age`. In a comment, say whether the distribution looks roughly symmetric or skewed, and why.

In [ ]:
# TODO: print mean, median, and the 0.1 and 0.9 quantiles of age; comment on the shape

In [ ]:
# @solution
print(df["age"].mean(), df["age"].median())
print(df["age"].quantile([0.1, 0.9]))
# Mean (~43) and median (41) are close, so the distribution is roughly symmetric,
# with a slightly longer tail towards older ages.

---

### Part 2: Group and Aggregate With `groupby()`

Suppose you want the average rating **for each department**. With what you know, you'd filter each department and compute the mean six times. `groupby()` does it in one step, in three stages usually called **split-apply-combine**:

1. **split** the rows into groups by the values of a column (`department`),
2. **apply** a calculation to each group (`mean` of `rating`),
3. **combine** the results into one Series, with one row per group.

In [ ]:
df.groupby("department")["rating"].mean()

Read it as *"group by department, take the rating column, compute the mean"*. The result is a Series whose index is the group names. Sort it to make the comparison obvious:

In [ ]:
df.groupby("department")["rating"].mean().sort_values()

`Trend` stands out with an average of 3.84, while the other five departments sit between about 4.14 and 4.28.

Before believing a difference like that, **always check how many rows each group has**. `.size()` counts the rows in each group:

In [ ]:
df.groupby("department").size()

`Trend` has only 118 reviews, compared with thousands for the others. That's still enough for a meaningful average, but it's worth reporting next to the result.

### Several Statistics at Once: `.agg()`

**`.agg()`** computes several statistics per group. Give it a list of names:

In [ ]:
df.groupby("department")["rating"].agg(["mean", "median", "count"]).round(2)

Or, for the most readable tables, use **named aggregation**: each keyword becomes a column name, and its value is a `(column, statistic)` pair. That way one table can summarise **different** columns:

In [ ]:
summary = df.groupby("department").agg(
    reviews=("rating", "size"),
    avg_rating=("rating", "mean"),
    share_recommended=("recommended", "mean"),
    median_words=("n_words", "median"),
)
summary.sort_values("avg_rating").round(2)

This is the kind of table that goes straight into a report. Useful statistic names: `"mean"`, `"median"`, `"std"`, `"min"`, `"max"`, `"sum"`, `"count"` (non-missing values), `"size"` (all rows), and `"nunique"`.

### Small Groups Can Mislead

Grouping by the more detailed `class_name` shows why group sizes matter:

In [ ]:
by_class = df.groupby("class_name").agg(reviews=("rating", "size"), avg_rating=("rating", "mean"))
by_class.sort_values("avg_rating").round(2).head(6)

`Casual bottoms` and `Chemises` have one review each, so their "average" is simply one customer's opinion. A common rule is to set a minimum group size before comparing, and filter the summary table like any DataFrame:

In [ ]:
by_class[by_class["reviews"] >= 100].sort_values("avg_rating").round(2)

### Grouping by Two Columns

Pass a list to group by two columns at once. The result has one row per **combination**:

In [ ]:
df.groupby(["division", "department"])["rating"].mean().round(2)

The two-level index is hard to read. **`.unstack()`** moves the inner level into columns, making a table:

In [ ]:
df.groupby(["division", "department"])["rating"].mean().unstack().round(2)

The `NaN` cells are combinations that don't exist in the data (the Intimates division only sells a few departments).

### Practice: `groupby()`

**Exercise 2: Recommendation by Division (easy)**

Compute the share of reviews that recommend the product in each `division`, sorted from lowest to highest, rounded to 3 decimals.

In [ ]:
# TODO: share of recommended by division, sorted, rounded to 3 decimals

In [ ]:
# @solution
df.groupby("division")["recommended"].mean().sort_values().round(3)

**Exercise 3: A Class Summary Table (medium)**

Build a table with one row per `class_name`, containing: the number of reviews (`reviews`), the average rating (`avg_rating`), the median number of helpful votes (`median_votes`), and the share of reviews mentioning a return (`share_return`). Keep only classes with at least 300 reviews, and sort by `share_return` from highest to lowest. Round to 3 decimals.

In [ ]:
# TODO: named aggregation by class_name, filter reviews >= 300, sort by share_return descending

In [ ]:
# @solution
class_summary = df.groupby("class_name").agg(
    reviews=("rating", "size"),
    avg_rating=("rating", "mean"),
    median_votes=("positive_feedback_count", "median"),
    share_return=("mentions_return", "mean"),
)
class_summary[class_summary["reviews"] >= 300].sort_values("share_return", ascending=False).round(3)

---

### Part 3: Crosstabs

To see how **two categorical columns** relate, count every combination. **`pd.crosstab(rows, columns)`** builds that table:

In [ ]:
pd.crosstab(df["department"], df["sentiment"])

Counts are hard to compare between departments of very different sizes. `normalize="index"` divides each **row** by its total, so each row shows shares that add up to 1:

In [ ]:
pd.crosstab(df["department"], df["sentiment"], normalize="index").round(3)

Now each row can be compared directly: about 10% of reviews are negative in most departments, but about 18% in `Trend`. (`normalize="columns"` would divide by column totals instead: *"of all negative reviews, what share come from each department?"*, a different question.)

### Turning Numbers Into Categories: `pd.cut()`

Is age related to satisfaction? Comparing 80 different ages is noisy; comparing a few **age groups** is clearer. **`pd.cut()`** assigns each value to a bin, given the bin edges and (optionally) labels:

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[17, 29, 39, 49, 59, 69, 100],
                         labels=["18-29", "30-39", "40-49", "50-59", "60-69", "70+"])

df.groupby("age_group", observed=True).agg(reviews=("rating", "size"),
                                           avg_rating=("rating", "mean")).round(2)

Each bin includes its **right** edge, so `(17, 29]` covers ages 18 to 29. (`observed=True` just tells pandas to list only the groups that actually occur; it silences a warning.)

The averages range only from about 4.14 to 4.29: age barely matters for the rating. **A variable that doesn't make a difference is also a finding**, and worth reporting.

### Practice: Crosstabs and Bins

**Exercise 4: Sentiment by Age Group (medium)**

Make a crosstab of `age_group` (rows) by `sentiment` (columns), normalized by row and rounded to 3 decimals. Which age group has the highest share of negative reviews? Answer in a comment.

In [ ]:
# TODO: crosstab of age_group by sentiment, normalized by row

In [ ]:
# @solution
pd.crosstab(df["age_group"], df["sentiment"], normalize="index").round(3)
# 70+ has the highest negative share (11.7%, but only 463 reviews), closely followed by 40-49 (11.3%).
# All groups are within about 9-12%, so age makes little difference.

---

### Part 4: The Dependent Variable

Every analysis needs a focus. In research and business analytics, that focus is the **dependent variable (DV)**: the outcome you want to understand, explain, or eventually predict. The other variables, the ones you use to explain it, are the **independent variables** (IVs), also called explanatory variables or features.

The names come from the question: the DV *depends on* the others. In *"Do longer reviews come with lower ratings?"*, the rating is the DV and review length is an IV.

| Research question | DV | Some IVs |
|---|---|---|
| What makes customers unhappy with a product? | rating / sentiment | department, review length, words used |
| Which reviews do other customers find helpful? | helpful votes | rating, length, age |
| Who recommends the product? | recommended | rating, department, age |

The same dataset can support different DVs, and each one leads to a different analysis. So the first real decision in an analysis is: **what is the DV?**

### Criteria for a Good DV

A good DV:

1. **Matches the question** you (or the business) care about.
2. **Varies enough**: if 99% of rows had the same value, there would be nothing to explain.
3. **Is measured reliably** for (almost) every row, with few missing values.
4. **Is not just a copy of another variable**: if two candidates carry the same information, pick one.

This dataset has three candidates. Check each one against the criteria.

**Candidate 1: `rating`** (ordinal, 1 to 5 stars)

In [ ]:
df["rating"].value_counts(normalize=True).sort_index().round(3)

It varies, it's never missing, and it directly measures satisfaction. Its weakness is the heavy tilt towards 5 stars (56%).

**Candidate 2: `recommended`** (binary, yes/no)

In [ ]:
print(df["recommended"].mean())
pd.crosstab(df["rating"], df["recommended"], normalize="index").round(3)

This crosstab is revealing. Almost nobody with 1 or 2 stars recommends the product, and almost everybody with 4 or 5 stars does. Only at **3 stars** is it split (about 41% recommend). `recommended` is very nearly a **simplified copy of the rating**, so it fails criterion 4: choosing both as DVs would mean analysing the same thing twice.

**Candidate 3: `positive_feedback_count`** (a count of helpful votes)

As you saw in Part 1, it's heavily right-skewed with about half zeros. More importantly, it answers a **different question**: not "is this customer satisfied?" but "do *other* customers find this review useful?". That's a perfectly good DV for a different study.

### Our Choice: Rating, Grouped Into Sentiment

For the rest of the course, the DV is **customer satisfaction**, measured by the `rating`. For many comparisons, it's clearer to use the three-level **`sentiment`** grouping you created in Lab 13:

* `negative` = 1 or 2 stars,
* `neutral` = 3 stars,
* `positive` = 4 or 5 stars.

Grouping the rating loses some detail (4 vs 5 stars), and in exchange makes groups large enough to compare and results easy to explain. You can always go back to the full 1-5 scale when the detail matters.

### Class Balance

When a DV is categorical, the first thing to check is how the rows are spread across its levels (its **classes**):

In [ ]:
df["sentiment"].value_counts(normalize=True).round(3)

The classes are strongly **imbalanced**: 77% positive, 12% neutral, and only 10% negative. This matters in two ways:

* **Averages over all reviews mostly describe happy customers.** The overall mean length or word frequency is dominated by the 77%. To learn about unhappy customers, always compare **by group**.
* **"Accuracy" can mislead.** A lazy rule that labels every review as positive would be "right" 77% of the time while being useless. In the follow-up course, when you build sentiment models, this 77% is the baseline any model has to beat.

About 2,300 negative reviews is still plenty for analysis. Imbalance isn't a problem to fix here; it's a property to keep in mind and report.

---

### Part 5: Exploring Variables Against the DV

With the DV chosen, the core of an exploratory analysis is comparing the other variables **across its levels**. One named aggregation does a lot of work:

In [ ]:
by_sentiment = df.groupby("sentiment").agg(
    reviews=("rating", "size"),
    avg_age=("age", "mean"),
    median_words=("n_words", "median"),
    avg_exclaims=("n_exclaims", "mean"),
    share_return=("mentions_return", "mean"),
    median_votes=("positive_feedback_count", "median"),
    avg_votes=("positive_feedback_count", "mean"),
)
by_sentiment.loc[["negative", "neutral", "positive"]].round(2)

(`.loc[[...]]` with a list of labels reorders the rows into a logical order, since `groupby` sorts them alphabetically.)

Read the table column by column, and you have a first set of findings:

* **Age**: no real difference. Satisfaction doesn't depend on the reviewer's age.
* **Length**: neutral reviews are a little longer, positive ones a little shorter. Customers with mixed feelings write the most, weighing pros and cons.
* **Exclamation marks**: positive reviews use about twice as many. Enthusiasm shows in punctuation (as you first saw on 600 reviews in Lab 09).
* **Returns**: about one in five negative reviews mentions returning the item, against about one in twenty-five positive reviews. One of the strongest signals in the table.
* **Helpful votes**: the medians are all 1 (skewed data again!), but the **means** differ: negative reviews collect more votes on average. Other shoppers seem to value warnings.

These are **descriptive** findings: they say what goes together, not what causes what. Exclamation marks don't *cause* happiness. But they are exactly the kind of evidence an EDA is meant to produce, and they point to what's worth charting (Lab 15) and investigating further (Lab 16).

### Formulating Analysis Questions

Good analysis questions are **specific**, **answerable with the data you have**, and **linked to the DV**. Compare:

| Too vague | Better |
|---|---|
| Are customers happy? | What share of reviews are negative, and does that share differ by department? |
| Does the text matter? | Do negative reviews mention returns, fit, or fabric more often than positive ones? |
| Is age important? | Does the average rating differ between age groups? |

Each "better" question tells you what to compute: a share by group, a comparison of keyword frequencies, a mean by bin. That's the test: **if you can't picture the table or chart that would answer it, the question isn't specific enough yet.**

### Practice: The DV

**Exercise 5: Height and Satisfaction (medium)**

`height_cm` is missing for reviews that don't mention a height. Create a Boolean column `mentions_height` (hint: `.notna()`), then compute the share of reviews that mention a height for each `sentiment`, in the order negative, neutral, positive.

In Lab 09, on only 600 reviews, height mentions looked roughly similar across ratings. What do you see on the full dataset? Answer in a comment, and suggest one reason for the difference.

In [ ]:
# TODO: add mentions_height, then compute its share by sentiment (negative, neutral, positive)

In [ ]:
# @solution
df["mentions_height"] = df["height_cm"].notna()
df.groupby("sentiment")["mentions_height"].mean().loc[["negative", "neutral", "positive"]].round(3)
# On the full data, negative reviews mention a height noticeably less often (about 13%)
# than positive ones (about 19%). Satisfied customers often add their measurements as
# advice ("I'm 5'4 and the XS fits"), while unhappy customers focus on what went wrong.
# 600 reviews (only ~60 negative) were too few to see this reliably.

**Exercise 6: Your Own Analysis Question (medium)**

Write one specific analysis question about the DV that hasn't been answered in this lab (as a comment), then write the code that answers it with `groupby()` or `pd.crosstab()`, and state the answer in one sentence.

*One possible example:* "Do reviews with a title have a different sentiment distribution from reviews without one?"

In [ ]:
# TODO: your question (as a comment), the code that answers it, and a one-sentence answer

In [ ]:
# @solution
# Question: do reviews with a title have a different sentiment distribution from reviews without one?
df["has_title"] = df["title"] != ""
pd.crosstab(df["has_title"], df["sentiment"], normalize="index").round(3)
# Answer: no; with or without a title, about 77% of reviews are positive and about 10% negative.

---

### Recap: Summarising and the DV

| Tool / Idea | What It Does | Example |
|---|---|---|
| `.mean()`, `.median()`, `.std()` | Centre and spread of a column | `df["n_words"].median()` |
| `.quantile(q)` | Value below which a share `q` falls | `df["age"].quantile([0.1, 0.9])` |
| Right-skewed | Long tail of large values; mean > median | helpful votes |
| `df.groupby(col)[x].mean()` | One statistic of `x` per group | `df.groupby("department")["rating"].mean()` |
| `.size()` | Rows per group | `df.groupby("department").size()` |
| `.agg(name=(col, stat))` | Several named statistics per group | `.agg(avg=("rating", "mean"))` |
| `groupby([a, b])` + `.unstack()` | Group by two columns, as a table | `.mean().unstack()` |
| `pd.crosstab(a, b, normalize="index")` | Counts (or row shares) of combinations | `pd.crosstab(df["department"], df["sentiment"])` |
| `pd.cut(col, bins, labels)` | Turn numbers into ordered categories | age groups |
| Dependent variable (DV) | The outcome the analysis is about | `sentiment` |
| Independent variables (IVs) | Variables used to explain the DV | length, department, ... |
| Class balance | Spread of a categorical DV across its levels | `value_counts(normalize=True)` |

The whole lab in one sentence: **`groupby` and `crosstab` compare statistics across groups, and choosing a dependent variable (here, sentiment from the star rating) turns those comparisons into an analysis with a clear question.**